# Task 1.1: Core Contribution / Architecture

**Paper:** Exact Discovery of Time Series Motifs  
**Authors:** Abdullah Mueen, Eamonn Keogh, Qiang Zhu, Sydney Cash, Brandon Westover  
**Venue:** KDD 2009 (ACM SIGKDD)  
**Student:** Abhishek (m23csai230137)

---

## Step-by-Step Method Description

The Mueen-Keogh (MK) algorithm is the first **exact** algorithm for discovering time series motifs that is significantly faster than brute-force search. A time series motif is defined as the pair of time series (or subsequences) in a database that are most similar to each other under Euclidean distance (Definition 3, Section 2). The algorithm works as follows:

### Step 1: Select Multiple Random Reference Time Series
- **Description:** The algorithm randomly picks R time series from the database D as reference points (ref₁, ref₂, ..., ref_R). For each reference, it computes the Euclidean distance from that reference to every other time series in D, storing these distances in a 2D table called `Dist` where `Dist[i][j] = d(refᵢ, Dⱼ)`.
- **Reference:** Table 3, Lines 2–7 (Section 3.2.2)
- **Purpose:** These pre-computed distances serve two purposes: (a) during computation, if any reference happens to be very close to some time series, the best-so-far is updated early (Line 6–7), giving us a tight initial bound; (b) the distances will later be used to compute lower bounds via the triangle inequality, allowing us to skip expensive true distance computations.

### Step 2: Select the Best Reference for Ordering
- **Description:** For each reference, the algorithm computes the standard deviation of its distance vector `Dist[i]`. The reference with the **largest** standard deviation is chosen as the "primary" reference for ordering the database. The indices of references are sorted into an ordering Z such that `S[Z(1)] >= S[Z(2)] >= ...`.
- **Reference:** Table 3, Lines 8–9 (Section 3.2.2)
- **Purpose:** A reference with high standard deviation in its distances creates a wider spread in the 1D projection. This means the lower bounds derived from it will be larger on average, allowing more candidate pairs to be pruned. The paper states: *"the larger the standard deviation is, the larger the lower bounds will be"* (Section 3.2.2).

### Step 3: Project and Sort the Database into a Linear Ordering
- **Description:** Using the primary reference Z(1), the algorithm sorts all m time series by their distance to this reference, producing an ordered index array I such that `Dist[Z(1), I(j)] <= Dist[Z(1), I(j+1)]`. This effectively projects all time series onto a 1D number line.
- **Reference:** Table 3, Line 10; Figure 3.B and 3.C (Section 3.1)
- **Purpose:** This linear ordering is the backbone of the search strategy. The key insight (Section 3.1) is: if two objects are close in the original high-dimensional space, they **must** also be close in this 1D ordering. Therefore, the true motif pair will appear nearby in this sorted list, and the algorithm can scan outward from small offsets.

### Step 4: Offset-Based Search with Triangle Inequality Pruning
- **Description:** The algorithm iterates over increasing `offset` values (1, 2, 3, ...). For each offset, it considers all pairs `{D[I(j)], D[I(j+offset)]}` for j = 1 to m−offset. Before computing the true distance, it checks the **lower bound** from every reference: `|Dist[Z(i), I(j)] - Dist[Z(i), I(j+offset)]|`. If **any** of these R lower bounds exceeds the current best-so-far, the pair is immediately **rejected** without computing the true distance.
- **Reference:** Table 3, Lines 11–21; Lemma 1 (Section 3.2.1)
- **Purpose:** This is where the main pruning happens. By using multiple references, the algorithm gets R independent lower bounds per pair. The triangle inequality guarantees `|d(ref, A) - d(ref, B)| <= d(A, B)`, so any lower bound exceeding best-so-far proves the pair cannot be the motif. The offset-based scan ensures **all** m(m−1)/2 pairs are covered (Lemma 2), guaranteeing exactness.

### Step 5: True Distance Computation with Early Abandoning
- **Description:** For pairs that survive all R lower bound checks, the algorithm computes the true Euclidean distance. However, it uses **early abandoning**: during the point-by-point squared difference accumulation, if the running sum exceeds the squared best-so-far at any point, the computation is immediately stopped.
- **Reference:** Table 3, Lines 22–25; Figure 2 (Section 2)
- **Purpose:** Early abandoning reduces the amortized cost of each distance computation from O(n) to much less. The paper explains (Section 4.3.1) that early abandoning is especially effective for motif discovery because the motif distance drops dramatically with database size (the "birthday paradox" analogy, Figure 9), so most non-motif pairs can be abandoned after checking only a few data points.

### Step 6: Termination Condition
- **Description:** For each offset value, the algorithm tracks whether **any** pair at that offset had its primary lower bound (from Z(1)) less than the best-so-far. If no pair at the current offset passes the primary lower bound check, the algorithm sets `abandon = true`. By **Lemma 1**, if no pair at offset k survives, then no pair at any offset > k will survive either, so the search can safely terminate.
- **Reference:** Table 3, Lines 12–13, 20–21; Lemma 1 (Section 3.2.1)
- **Purpose:** This is what makes the algorithm finish far before exhausting all O(m²) pairs. In practice, the algorithm terminates after checking only a small fraction of all pairs, achieving up to three orders of magnitude speedup over brute force (Section 4.1, Figure 6).

### Step 7: Output
- **Description:** The algorithm returns the indices L1 and L2 of the motif pair, i.e., the two time series in D with the minimum Euclidean distance between them.
- **Reference:** Table 3, Line 25
- **Purpose:** This is the exact motif — guaranteed to be the globally closest pair in the database.

## Final Summary Sentence

This paper solves the problem of finding the **exact closest pair (motif)** in a database of time series — a problem previously considered intractable for large datasets — by using triangle inequality-based lower bounds from multiple random reference points combined with an offset-based ordered search strategy and early abandoning, achieving up to **three orders of magnitude speedup** over brute-force search while guaranteeing the same exact result (Section 1, Abstract).